# Gym Workout Recommendation System
Coursework walkthrough grounded in the supplied presentation and chapters 1?5, 7 and 8. Chapter 6 is missing; collaborative filtering follows the topics named in the later chapter outlines. Run this notebook from the project folder after preparing data.

In [1]:
from pathlib import Path
import pandas as pd
from engine import Profile, Recommender, METHODS, ROOT, make_plan, comparable_sessions
from storage import activities
items = pd.read_csv(ROOT / "data/processed/exercises.csv")
members = pd.read_csv(ROOT / "data/processed/members.csv")
print("Exercises:", len(items), "Source sessions:", len(members))
items.head()

Exercises: 2872 Source sessions: 973
                    exercise_id  ... instruction_source
0        ex_partnerplankbandrow  ...                NaN
1  ex_bandedcrunchisometrichold  ...                NaN
2         ex_fyrbandedplankjack  ...                NaN
3               ex_bandedcrunch  ...                NaN
4                     ex_crunch  ...                NaN

[5 rows x 11 columns]


## Chapters 1?2: data, preprocessing and exploration
Raw downloaded files, normalized titles, missing source ratings and exact instruction matches are retained for audit. Member sessions have no exercise IDs and cannot be used as a rating matrix.

In [2]:
display(items.isna().sum().to_frame("missing"))
display(items.groupby(["level", "category"]).size().to_frame("exercises"))
display(members.describe())

                    missing
exercise_id               0
name                      0
description               0
category                  0
muscle                    0
equipment                 0
required_equipment        0
level                     0
source_rating          1860
instructions           2292
instruction_source     2292
                                    exercises
level        category                        
beginner     cardio                         9
             olympic weightlifting         30
             plyometrics                   35
             powerlifting                  27
             strength                     276
             stretching                    63
             strongman                     16
expert       plyometrics                    2
             strength                      10
intermediate cardio                        25
             olympic weightlifting          4
             plyometrics                   60
             powerli

## Chapters 3?4: popularity prior and cosine similarity
The source catalogue lacks rating counts. The popularity baseline uses source scores with local feedback shrinkage. Content cosine uses exercise metadata and the active profile.

In [3]:
profile = Profile(goal="Muscle Gain", experience="intermediate", equipment=["bodyweight", "dumbbell", "bench"], minutes=30)
engine = Recommender(history=activities())
for method in ["Popularity", "Content cosine"]:
    rec, info = engine.recommend(profile, method, top_n=5)
    print(method, info)
    display(rec[["name", "muscle", "score"]])

Popularity {'eligible': 1539, 'collaborative_ready': False, 'fallback': False, 'message': ''}
                                    name      muscle  score
0  Dumbbell front raise to lateral raise   shoulders   0.95
1                   Incline Hammer Curls      biceps   0.95
2       Romanian Deadlift With Dumbbells  hamstrings   0.94
3                            Triceps dip     triceps   0.94
4                             Bottoms Up  abdominals   0.93
Content cosine {'eligible': 1539, 'collaborative_ready': False, 'fallback': False, 'message': ''}
                                              name       muscle     score
0                           Decline Dumbbell Flyes        chest  0.157589
1                                        Muscle Up         lats  0.093813
2                       Decline dumbbell chest fly  middle back  0.091932
3         Incline Dumbbell Curl - Gethin Variation       biceps  0.073789
4  Incline Front Dumbbell Raise - Gethin Variation    shoulders  0.072810


## Chapters 5?6: SVD and collaborative filtering
These methods use genuine locally recorded ratings. A new installation has none, so an explicit hybrid fallback is expected. Deterministic synthetic fixtures are used only in unit tests, not as downloaded user data or reported accuracy.

In [4]:
for method in ["SVD", "User CF", "Item CF"]:
    rec, info = engine.recommend(profile, method, top_n=5)
    print(method, info)
    display(rec[["name", "score"]])

SVD {'eligible': 1539, 'collaborative_ready': False, 'fallback': True, 'message': 'Hybrid fallback: collaborative methods need this user to rate at least 2 exercises and at least 2 users overall.'}
                          name     score
0       Decline Dumbbell Flyes  0.613036
1                    Muscle Up  0.585525
2  Single-dumbbell front raise  0.572815
3     Dumbbell V-Sit Cross Jab  0.571531
4     Tricep Dumbbell Kickback  0.570969
User CF {'eligible': 1539, 'collaborative_ready': False, 'fallback': True, 'message': 'Hybrid fallback: collaborative methods need this user to rate at least 2 exercises and at least 2 users overall.'}
                          name     score
0       Decline Dumbbell Flyes  0.613036
1                    Muscle Up  0.585525
2  Single-dumbbell front raise  0.572815
3     Dumbbell V-Sit Cross Jab  0.571531
4     Tricep Dumbbell Kickback  0.570969
Item CF {'eligible': 1539, 'collaborative_ready': False, 'fallback': True, 'message': 'Hybrid fallback: coll

## Chapter 7: hard constraints and cases
Equipment requirements, experience limits and primary muscle exclusions filter candidates before ranking. Similar source sessions are retrieved using standardized age, height, weight and experience; source calories are descriptive only.

In [5]:
cases = comparable_sessions(profile, members)
cases[["Workout_Type", "Calories_Burned", "Session_Duration (hours)", "similarity"]].head(10)

    Workout_Type  Calories_Burned  Session_Duration (hours)  similarity
374         Yoga           1123.0                      1.43    0.901977
599     Strength           1022.0                      1.46    0.852582
918       Cardio            831.0                      1.13    0.845029
501       Cardio            907.0                      1.08    0.844261
822       Cardio            756.0                      1.19    0.838958
213     Strength           1043.0                      1.28    0.825644
740     Strength            938.0                      1.36    0.809578
579         HIIT           1016.0                      1.27    0.803676
210         HIIT           1147.0                      1.47    0.801101
252     Strength           1056.0                      1.39    0.784716


## Chapter 8: current context and a bounded session
Changing available time, location and energy changes eligibility and allocation. Duration blocks are transparent project defaults, not medically validated prescriptions.

In [6]:
ranked, info = engine.recommend(profile)
plan = make_plan(ranked, profile)
display(plan[["name", "muscle", "duration_minutes", "reason"]])
print("Allocated with preparation:", int(plan.duration_minutes.sum()) + 5, "minutes")

                          name  ...                                             reason
0       Decline Dumbbell Flyes  ...  chest; intermediate; available equipment; musc...
1                    Muscle Up  ...  lats; intermediate; available equipment; muscl...
2  Single-dumbbell front raise  ...  shoulders; intermediate; available equipment; ...

[3 rows x 4 columns]
Allocated with preparation: 26 minutes


## Evaluation
Scenario checks report feasibility, diversity and coverage on the real catalogue. Ranking accuracy is unavailable until sufficient genuine histories exist. Evaluation uses pre-cutoff training and a held-out unseen positive to avoid future feedback leakage.

In [7]:
import evaluate
evaluate.main()
import json
json.loads((ROOT / "reports/evaluation.json").read_text())

{
  "catalogue_items": 2872,
  "scenarios": 36,
  "budget_violations": 0,
  "equipment_violations": 0,
  "scenario_catalogue_coverage": 0.006267409470752089,
  "ranking_evaluation": {
    "status": "unavailable: no eligible genuine held-out histories; no synthetic accuracy is reported"
  },
  "limitations": "Scenario coverage describes only tested profiles. No evidence of fitness outcomes or real-world ranking accuracy without adequate genuine feedback."
}
{'catalogue_items': 2872, 'scenarios': 36, 'budget_violations': 0, 'equipment_violations': 0, 'scenario_catalogue_coverage': 0.006267409470752089, 'ranking_evaluation': {'status': 'unavailable: no eligible genuine held-out histories; no synthetic accuracy is reported'}, 'limitations': 'Scenario coverage describes only tested profiles. No evidence of fitness outcomes or real-world ranking accuracy without adequate genuine feedback.'}
